<a href="https://colab.research.google.com/github/AntonyJohny/Innomatics_Research_Labs/blob/main/IN126001202_GenAI/Task_5/Task_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning DistilBERT for Token Classification (Chunking)

## Internship Task – NLP Assignment 5  
This project implements a transformer-based model for token classification using DistilBERT to perform Chunking (Phrase Detection).

In [22]:
!pip install transformers datasets seqeval evaluate accelerate nltk -q

In [3]:
import nltk
from nltk.corpus import conll2000
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import precision_score, recall_score, f1_score

In [4]:
nltk.download('conll2000', quiet=True)

True


### Dataset Used: CoNLL-2000 (via NLTK)

- Task: Chunking (Phrase Detection)
- Labels: NP (Noun Phrase), VP (Verb Phrase), PP (Prepositional Phrase)

Due to compatibility issues with Hugging Face dataset scripts, the dataset was loaded using NLTK.

In [5]:
train_data = conll2000.chunked_sents('train.txt', chunk_types=['NP', 'VP', 'PP'])
test_data = conll2000.chunked_sents('test.txt', chunk_types=['NP', 'VP', 'PP'])


### Data Preprocessing

- Extract tokens and chunk labels
- Convert to sequence format
- Prepare for token classification

In [6]:
def extract_data(dataset):
    tokens = []
    labels = []

    for sent in dataset:
        words = []
        tags = []

        for word, pos, chunk in nltk.chunk.tree2conlltags(sent):
            words.append(word)
            tags.append(chunk)

        tokens.append(words)
        labels.append(tags)

    return tokens, labels

train_tokens, train_labels = extract_data(train_data)
test_tokens, test_labels = extract_data(test_data)


### Label Mapping

Convert text labels into numerical format for model training.

In [7]:
label_list = list(set(label for sublist in train_labels for label in sublist))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print(label_list)

['I-VP', 'O', 'I-PP', 'B-NP', 'B-VP', 'I-NP', 'B-PP']


### Convert to Hugging Face Dataset Format

In [8]:
train_dataset = Dataset.from_dict({
    "tokens": train_tokens,
    "tags": [[label2id[l] for l in seq] for seq in train_labels]
})

test_dataset = Dataset.from_dict({
    "tokens": test_tokens,
    "tags": [[label2id[l] for l in seq] for seq in test_labels]
})

dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8936
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 2012
    })
})


### Tokenization & Label Alignment

- Use DistilBERT tokenizer
- Handle subwords
- Assign -100 to ignored tokens

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [10]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

### Model Setup

Using DistilBERT for token classification.

In [11]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/8936 [00:00<?, ? examples/s]

Map:   0%|          | 0/2012 [00:00<?, ? examples/s]

### Model Setup

Using DistilBERT for token classification.

In [12]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Training Configuration

In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",   # ✅ fixed
    logging_steps=50
)

### Evaluation Metrics

Using Precision, Recall, and F1 Score.

In [14]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [id2label[l] for l in label if l != -100]
        for label in labels
    ]

    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

### Trainer Initialization

In [15]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(2000)),
    eval_dataset=tokenized_datasets["test"].select(range(500)),
    data_collator=data_collator,   # ✅ THIS FIXES ERROR
    compute_metrics=compute_metrics
)

### Model Training

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.382556,0.158597,0.921540,0.948943,0.935041
2,0.156917,0.127323,0.937006,0.954180,0.945515


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.42527848625183107, metrics={'train_runtime': 40.2782, 'train_samples_per_second': 99.309, 'train_steps_per_second': 6.207, 'total_flos': 58786960506912.0, 'train_loss': 0.42527848625183107, 'epoch': 2.0})

### Model Evaluation

In [17]:
trainer.evaluate()

{'eval_loss': 0.12732268869876862,
 'eval_precision': 0.9370064279155188,
 'eval_recall': 0.9541799139704508,
 'eval_f1': 0.9455151964418088,
 'eval_runtime': 1.058,
 'eval_samples_per_second': 472.594,
 'eval_steps_per_second': 30.246,
 'epoch': 2.0}

### Inference on Custom Sentence

In [20]:
import torch

def predict(sentence):
    tokens = sentence.split()

    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    # ✅ Move inputs to same device as model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model(**inputs)
    predictions = outputs.logits.argmax(dim=2)

    print("\nPredictions:")
    for token, pred in zip(tokens, predictions[0][1:len(tokens)+1]):
        print(f"{token} → {id2label[pred.item()]}")

In [21]:
predict("John works at Google in California")


Predictions:
John → B-NP
works → B-VP
at → B-PP
Google → B-NP
in → B-PP
California → B-NP


### Comparison
### POS Tagging vs Chunking

- POS Tagging identifies grammatical roles (noun, verb, adjective)
- Chunking identifies phrases (NP, VP, PP)
- Chunking is more complex and context-aware

### Challenges Faced

- Handling subword tokenization
- Aligning labels with tokens
- Dataset compatibility issues


### Observations

- Transformer models capture context effectively
- Chunking requires deeper contextual understanding than POS tagging